# NB01 — Core Modules

Defines the model + metric building blocks used by every later notebook (plan v10 §4) and
writes them to `core_modules.py` (plus the dataset code to `dental_dataset.py`) at the
project root so `NB02`–`NB08` can simply `from core_modules import SwinGazeModel`.

**What this notebook produces**
- `core_modules.py` — `ConvBlock`, `SpatialDecoder`, `SwinGazeModel`,
  `align_attention_to_gaze`, `compute_dice`, `compute_metrics`, `evaluate_model`
- `dental_dataset.py` — `DentalDataset`, `make_weighted_sampler`, `build_dataloaders`

The final section runs a smoke test (forward pass, shape checks, LoRA %, gaze-KL integration).

**Prerequisite:** `config.json` at project root (created in NB00 setup).


In [7]:
import os, sys, json

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT = '/content/drive/MyDrive/DENTAL'
except ImportError:
    # Running locally on Windows or Linux
    cwd = os.getcwd()
    PROJECT = os.path.abspath(os.path.join(cwd, '..')) if os.path.basename(cwd) == 'Notebooks' else cwd

NOTEBOOKS_DIR = os.path.join(PROJECT, 'Notebooks')
sys.path.insert(0, PROJECT)
sys.path.insert(0, NOTEBOOKS_DIR)
os.chdir(PROJECT)

with open('config.json', 'r') as f:
    CONFIG = json.load(f)

print('Working directory:', os.getcwd())
print('Model:', CONFIG['model_name'])
print('LoRA rank/alpha:', CONFIG['lora_rank'], CONFIG.get('lora_alpha', 16), '| modules:', CONFIG['lora_modules'])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/DENTAL
Model: microsoft/swin-tiny-patch4-window7-224
LoRA rank/alpha: 8 16 | modules: ['q_proj', 'v_proj']


In [8]:
import os, sys, json

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT = '/content/drive/MyDrive/DENTAL'
except ImportError:
    # Running locally on Windows or Linux
    cwd = os.getcwd()
    PROJECT = os.path.abspath(os.path.join(cwd, '..')) if os.path.basename(cwd) == 'Notebooks' else cwd

NOTEBOOKS_DIR = os.path.join(PROJECT, 'Notebooks')
sys.path.insert(0, PROJECT)
sys.path.insert(0, NOTEBOOKS_DIR)
os.chdir(PROJECT)

with open('config.json', 'r') as f:
    CONFIG = json.load(f)

print('Working directory:', os.getcwd())
print('Model:', CONFIG['model_name'])
print('LoRA rank/alpha:', CONFIG['lora_rank'], CONFIG.get('lora_alpha', 16), '| modules:', CONFIG['lora_modules'])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/DENTAL
Model: microsoft/swin-tiny-patch4-window7-224
LoRA rank/alpha: 8 16 | modules: ['q_proj', 'v_proj']


In [9]:
!pip install --upgrade "torchao>=0.16.0"
!pip install -q torch torchvision transformers peft scikit-learn numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [10]:
%%writefile core_modules.py
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import SwinModel
from peft import LoraConfig, get_peft_model
from sklearn.metrics import f1_score, recall_score, roc_auc_score

# ==========================================
# 1. Architecture: SwinGazeModel + Decoder
# ==========================================

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class SpatialDecoder(nn.Module):
    def __init__(self, hidden_dims=[96, 192, 384, 768], out_classes=1):
        super().__init__()
        self.up4   = nn.ConvTranspose2d(hidden_dims[3], hidden_dims[2], 2, stride=2)
        self.conv3 = ConvBlock(hidden_dims[2] * 2, hidden_dims[2])
        self.up3   = nn.ConvTranspose2d(hidden_dims[2], hidden_dims[1], 2, stride=2)
        self.conv2 = ConvBlock(hidden_dims[1] * 2, hidden_dims[1])
        self.up2   = nn.ConvTranspose2d(hidden_dims[1], hidden_dims[0], 2, stride=2)
        self.conv1 = ConvBlock(hidden_dims[0] * 2, hidden_dims[0])
        self.final_up = nn.Sequential(
            nn.ConvTranspose2d(hidden_dims[0], 32, 4, stride=4),
            nn.Conv2d(32, out_classes, 1)
        )

    def forward(self, features):
        f1, f2, f3, f4 = features
        def reshape_state(f):
            B, L, C = f.shape
            H = int(math.sqrt(L))
            assert H * H == L, f"Feature seq len {L} is not a perfect square; H={H}"
            return f.reshape(B, H, H, C).permute(0, 3, 1, 2).contiguous()
        f1 = reshape_state(f1)   # [B,  96, 56, 56]
        f2 = reshape_state(f2)   # [B, 192, 28, 28]
        f3 = reshape_state(f3)   # [B, 384, 14, 14]
        f4 = reshape_state(f4)   # [B, 768,  7,  7]
        x = self.up4(f4)
        x = self.conv3(torch.cat([x, f3], dim=1))
        x = self.up3(x)
        x = self.conv2(torch.cat([x, f2], dim=1))
        x = self.up2(x)
        x = self.conv1(torch.cat([x, f1], dim=1))
        return self.final_up(x)

class SwinGazeModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        base = SwinModel.from_pretrained(config['model_name'], attn_implementation="eager")
        self._use_lora = config.get('use_lora', True)  # [v21 R1] needed by forward() to route correctly

        if self._use_lora:
            lora_cfg = LoraConfig(
                r=config['lora_rank'],
                lora_alpha=config.get('lora_alpha', 16),
                target_modules=config['lora_modules']
            )
            self.encoder = get_peft_model(base, lora_cfg)
        else:
            self.encoder = base
            print(" [INFO] use_lora=False: running no-LoRA baseline")

        for name, param in self.encoder.named_parameters():
            if 'lora' in name:
                param.requires_grad = True
            elif 'layers.3' in name:       # unfreeze last Swin stage for optimal feature adaptation
                param.requires_grad = True
            else:
                param.requires_grad = False

        self.decoder = SpatialDecoder()
        self.cls_head = nn.Sequential(
            nn.LayerNorm(768),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def verify_lora_applied(self):
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        enc_trainable = sum(p.numel() for n, p in self.encoder.named_parameters() if p.requires_grad)
        dec_trainable = sum(p.numel() for p in self.decoder.parameters() if p.requires_grad)
        cls_trainable = sum(p.numel() for p in self.cls_head.parameters() if p.requires_grad)
        ratio = trainable / total * 100
        assert ratio > 0.0, "FATAL: 0% trainable parameters. Check LoRA target_modules."
        print(f" LoRA applied: {trainable:,} / {total:,} trainable ({ratio:.3f}%)")
        print(f"  encoder: {enc_trainable:,}  decoder: {dec_trainable:,}  cls_head: {cls_trainable:,}")
        return ratio

    def forward(self, x):
        # [v21 R1] Branch on use_lora — plain SwinModel has no .base_model attribute.
        if self._use_lora:
            out = self.encoder.base_model(
                pixel_values=x,
                output_attentions=True,
                output_hidden_states=True
            )
        else:
            out = self.encoder(
                pixel_values=x,
                output_attentions=True,
                output_hidden_states=True
            )
        features = [out.hidden_states[i] for i in (0, 1, 2, 4)]
        seg_map  = self.decoder(features)                             # [B, 1, 224, 224]
        cls_features = out.hidden_states[4]                           # [B, 49, 768]
        cls_logits = self.cls_head(cls_features.mean(dim=1))          # [B, 1] classification logits
        return cls_logits, seg_map, out

# ==========================================
# 2. Alignment, Loss, and Metric Utilities
# ==========================================

def align_attention_to_gaze(attn_map, target_size=48):
    # Interpolates [B, 7, 7] or spatial attention map to [B, target_size, target_size].
    if attn_map.dim() == 3:
        attn_map = attn_map.unsqueeze(1)
    aligned = F.interpolate(attn_map, size=(target_size, target_size),
                            mode="bilinear", align_corners=False)
    return aligned.squeeze(1)

def compute_dice(pred, target):
    # Dice score for binary numpy arrays.
    pred   = pred.astype(bool)
    target = target.astype(bool)
    intersection = (pred & target).sum()
    return (2. * intersection) / (pred.sum() + target.sum() + 1e-5)

def compute_metrics(targets, predictions, probs):
    metrics = {}
    metrics['Dice']   = compute_dice(predictions, targets)
    metrics['F1']     = f1_score(targets, predictions, zero_division=0)
    metrics['Recall'] = recall_score(targets, predictions, zero_division=0)
    try:
        metrics['AUROC'] = roc_auc_score(targets, probs)
    except ValueError:
        metrics['AUROC'] = 0.5
    return metrics

def evaluate_model(model, loader, device):
    # Run inference on loader; return metrics dict with Dice, F1, Recall, AUROC.
    model.eval()
    all_preds, all_targets, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            logits, _, _, _ = model(batch['image'].to(device))
            probs = torch.sigmoid(logits.reshape(-1)).cpu().numpy()
            preds = (probs > 0.5).astype(int)
            all_probs.extend(probs.tolist())
            all_preds.extend(preds.tolist())
            all_targets.extend(batch['has_lesion_gt'].numpy().tolist())
    model.train()
    return compute_metrics(
        np.array(all_targets),
        np.array(all_preds),
        np.array(all_probs)
    )


Overwriting core_modules.py


### Write `dental_dataset.py`
Unifies dataset loading and KL-divergence safe augmentations across the pipeline.


In [11]:
%%writefile dental_dataset.py
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T

class DentalDataset(Dataset):
    # Loads dental radiographs + optional 48x48 expert gaze density map.
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    def __init__(self, csv_path, project_root, augment=False, image_root=None):
        self.df           = pd.read_csv(csv_path)
        self.root         = project_root
        self.image_root   = image_root or project_root
        self.augment      = augment
        self.to_tensor    = T.ToTensor()
        self.normalise    = T.Normalize(mean=self.MEAN, std=self.STD)

        if self.augment:
            self.geom_tf = T.RandomChoice([
                T.RandomHorizontalFlip(p=1.0),
                T.RandomRotation(degrees=15),
                T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
                T.RandomAffine(degrees=0, scale=(0.95, 1.05)),
            ])
        else:
            self.geom_tf = None

    def __len__(self):
        return len(self.df)

    def _load_image(self, path):
        # Load image as RGB PIL, resize to 224x224.
        full = path if os.path.isabs(path) else os.path.join(self.image_root, path)
        img = Image.open(full).convert('RGB')
        return img.resize((224, 224), Image.Resampling.BILINEAR)

    def _load_gaze(self, path):
        # Load float32 48x48 npy array, normalise sum to 1.0.
        full = path if os.path.isabs(path) else os.path.join(self.root, path)
        if not os.path.exists(full):
            return np.zeros((48, 48), dtype=np.float32)
        arr = np.load(full).astype(np.float32)
        s = arr.sum()
        if s > 0:
            arr /= s
        return arr

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = self._load_image(row['image_path'])
        has_gaze = int(row.get('has_gaze_map', 0))
        gaze = self._load_gaze(row['gaze_map_path']) if has_gaze else np.zeros((48, 48), dtype=np.float32)

        if self.augment and self.geom_tf is not None:
            seed = torch.random.initial_seed()
            torch.manual_seed(seed)
            img_t = self.to_tensor(img)
            img_t = self.geom_tf(img_t)
            torch.manual_seed(seed)
            gaze_t = torch.from_numpy(gaze).unsqueeze(0)
            gaze_t = self.geom_tf(gaze_t).squeeze(0).numpy()
            s = gaze_t.sum()
            if s > 0:
                gaze_t /= s
            gaze = gaze_t
        else:
            img_t = self.to_tensor(img)

        img_t = self.normalise(img_t)
        has_lesion = int(row['has_lesion_gt'])
        label = torch.tensor(has_lesion, dtype=torch.float32)
        gaze_tensor = torch.from_numpy(gaze).unsqueeze(0)
        has_gaze_tensor = torch.tensor(has_gaze, dtype=torch.float32)

        return {
            'image':         img_t,
            'has_lesion_gt': label,
            'gaze_map':      gaze_tensor,
            'has_gaze_map':  has_gaze_tensor,
            'image_id':      str(row['id']),
            'dataset':       str(row.get('dataset', 'unknown'))
        }

def make_weighted_sampler(df):
    # Creates WeightedRandomSampler to balance lesion positive/negative classes.
    labels  = df['has_lesion_gt'].values
    n_total = len(labels)
    n_pos   = (labels == 1).sum()
    n_neg   = (labels == 0).sum()
    w_pos   = n_total / (2.0 * max(n_pos, 1))
    w_neg   = n_total / (2.0 * max(n_neg, 1))
    weights = [w_pos if l == 1 else w_neg for l in labels]
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

def build_dataloaders(CONFIG, project_root, batch_size=None, fold=None, smoke_test=False, image_root=None):
    bs = batch_size or CONFIG['chosen_batch_size']
    img_root = image_root or project_root
    train_df = pd.read_csv(os.path.join(project_root, 'splits/train_split.csv'))

    train_ds = DentalDataset(os.path.join(project_root, 'splits/train_split.csv'),
                             project_root, augment=True, image_root=img_root)
    val_ds   = DentalDataset(os.path.join(project_root, 'splits/val_split.csv'),
                             project_root, augment=False, image_root=img_root)
    sampler  = make_weighted_sampler(train_df)

    test_ds    = DentalDataset(os.path.join(project_root, 'splits/test_split.csv'),
                               project_root, augment=False, image_root=img_root)
    dentex_ds  = DentalDataset(os.path.join(project_root, 'splits/dentex_test_split.csv'),
                               project_root, augment=False, image_root=img_root)

    oral_test_csv = os.path.join(project_root, 'splits/oralxray_test_split.csv')
    oral_test_ds  = DentalDataset(oral_test_csv, project_root, augment=False, image_root=img_root) if os.path.exists(oral_test_csv) else None

    oral_train_csv = os.path.join(project_root, 'splits/oralxray_train_split.csv')
    oral_train_ds  = DentalDataset(oral_train_csv, project_root, augment=True, image_root=img_root) if os.path.exists(oral_train_csv) else None
    oral_sampler   = make_weighted_sampler(pd.read_csv(oral_train_csv)) if os.path.exists(oral_train_csv) else None

    sweep_csv = os.path.join(project_root, 'splits/sweep_split.csv')
    sweep_ds  = DentalDataset(sweep_csv, project_root, augment=False, image_root=img_root) if os.path.exists(sweep_csv) else train_ds
    sweep_df_ref  = pd.read_csv(sweep_csv) if os.path.exists(sweep_csv) else train_df
    sweep_sampler = make_weighted_sampler(sweep_df_ref)

    # Use num_workers=0 on local Windows scripts to prevent multiprocessing locks, or 2 if protected
    nw = 0 if os.name == 'nt' else 2

    loaders = {
        'train':          DataLoader(train_ds,  batch_size=bs, sampler=sampler,
                                     num_workers=nw, pin_memory=True, drop_last=True),
        'val':            DataLoader(val_ds,    batch_size=bs, shuffle=False,
                                     num_workers=nw, pin_memory=True),
        'test_tufts':     DataLoader(test_ds,   batch_size=bs, shuffle=False,
                                     num_workers=nw, pin_memory=True),
        'dentex':         DataLoader(dentex_ds, batch_size=bs, shuffle=False,
                                     num_workers=nw, pin_memory=True),
        'sweep':          DataLoader(sweep_ds,  batch_size=bs, sampler=sweep_sampler,
                                     num_workers=nw, pin_memory=True, drop_last=True),
        'oralxray_train': DataLoader(oral_train_ds, batch_size=bs, sampler=oral_sampler,
                                     num_workers=nw, pin_memory=True, drop_last=True) if oral_train_ds else None,
        'test_oralxray':  DataLoader(oral_test_ds, batch_size=bs, shuffle=False,
                                     num_workers=nw, pin_memory=True) if oral_test_ds else None,
    }
    return loaders


Overwriting dental_dataset.py


## 4.1 — `SwinGazeModel` + `SpatialDecoder`

Swin-T encoder (frozen) with LoRA adapters on `query`/`value` (HuggingFace Swin-T layer
names), plus a U-Net-style `SpatialDecoder` over the 4 multi-scale Swin hidden states.

**[v18 E1]** `SwinModel.hidden_states` returns 5 tensors; the decoder needs the four
feature maps at channels `[96, 192, 384, 768]`, so we cherry-pick indices `(0, 1, 2, 4)`
(index 3 is skipped — it is the same 7×7 spatial size as index 4).


## 4.2 — Alignment & metrics

`align_attention_to_gaze` upsamples the Swin attention grid to the gaze resolution;
`compute_metrics` returns the Dice / F1 / Recall / AUROC dict used across evaluation notebooks.


## Smoke test

Builds the model from `config.json`, runs a forward pass, and verifies:
- decoder logits are `[B, 1, 224, 224]`
- last-stage attention has 49 key tokens (7×7 window) — the tensor used for the gaze-KL term
- LoRA + Decoder trainable fraction (~16 %)
- the attention → 48×48 alignment → KL pipeline produces a finite loss

**Image-level note:** the decoder emits a spatial map. Downstream classification logits are
obtained by pooling, e.g. `cls_logit = logits.mean(dim=(2, 3)).squeeze(1)` — this is the
image-level value `NB02`/`NB03` feed to `binary_cross_entropy_with_logits`.


In [12]:
import importlib
import numpy as np
import torch
import torch.nn.functional as F

import core_modules
importlib.reload(core_modules)
from core_modules import SwinGazeModel, align_attention_to_gaze, compute_metrics

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

model = SwinGazeModel(CONFIG).to(device)
model.eval()

B = 2
x = torch.randn(B, 3, CONFIG['chosen_resolution'], CONFIG['chosen_resolution']).to(device)
with torch.no_grad():
    logits, seg_map, last_attn, out = model(x)

print('logits shape:', tuple(logits.shape))
assert logits.shape == (B, 1), f'unexpected logits shape: {logits.shape}'

print('seg_map shape:', tuple(seg_map.shape))
assert seg_map.shape == (B, 1, 224, 224), f'unexpected decoder output shape: {seg_map.shape}'

last_attn_raw = out.attentions[-1]
print('last attention shape:', tuple(last_attn_raw.shape))
assert last_attn_raw.shape[-1] == 49, f'expected 49 key tokens, got {last_attn_raw.shape[-1]}'


Device: cpu


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

[transformers] SwinModel LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


logits shape: (2, 1)
seg_map shape: (2, 1, 224, 224)
last attention shape: (2, 24, 49, 49)


In [13]:
# LoRA trainable fraction
model.verify_lora_applied()


 LoRA applied: 19,606,994 / 32,942,492 trainable (59.519%)
  encoder: 14,325,168  decoder: 5,083,169  cls_head: 198,657


59.51885485773207

In [14]:
# Attention -> 48x48 alignment -> gaze KL integration (matches NB02 §6.1)
last_attn = last_attn_raw.mean(dim=1).mean(dim=1).reshape(-1, 7, 7)
spatial_attn_48 = align_attention_to_gaze(last_attn, 48)

dummy_gaze = torch.rand(B, 48, 48, device=device)
valid_attn = spatial_attn_48 / (spatial_attn_48.sum(dim=(1, 2), keepdim=True) + 1e-9)
valid_gaze = dummy_gaze / (dummy_gaze.sum(dim=(1, 2), keepdim=True) + 1e-9)
gaze_kl = F.kl_div((valid_attn + 1e-9).log(), valid_gaze, reduction='batchmean')

print('aligned attention shape:', tuple(spatial_attn_48.shape))
print('gaze KL:', float(gaze_kl))
assert torch.isfinite(gaze_kl), 'gaze KL is not finite'
assert spatial_attn_48.shape == (B, 48, 48)


aligned attention shape: (2, 48, 48)
gaze KL: 0.6047965288162231


In [15]:
# compute_metrics sanity check on dummy predictions
targets = np.array([0, 1, 1, 0, 1])
preds   = np.array([0, 1, 0, 0, 1])
probs   = np.array([0.1, 0.9, 0.4, 0.2, 0.8])
print(compute_metrics(targets, preds, probs))

print('\nNB01 smoke test passed — core_modules.py ready for NB02_stage1_train.ipynb')


{'Dice': np.float64(0.7999984000032001), 'F1': 0.8, 'Recall': 0.6666666666666666, 'AUROC': np.float64(1.0)}

NB01 smoke test passed — core_modules.py ready for NB02_stage1_train.ipynb
